In [1]:
import sys
import numpy as np

sys.path.append("../../../")
from Rain import Rain
sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-02 20:52:11.593305: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-02 20:52:12.375160: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import sys
sys.path.append('../../../')
from clean_all import clean
clean()
sys.path.pop()

'../../../'

In [3]:
config = {
    "lib": "tensorflow",
    "mode": 'local',
    "partitions": 3,
    "iterations": 3,
    "lr": 0.001,
    "epochs": 2,
    "batch_size": 128,
    "loss": tf.keras.losses.CategoricalCrossentropy(),
    "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )


def partition_train_data(X_train, y_train, partitions):
    num_samples = X_train.shape[0]

    # Create an array of indices from 0 to num_samples - 1
    indices = np.arange(num_samples)

    # Shuffle the indices
    np.random.shuffle(indices)

    # Use the shuffled indices to shuffle the datasets
    X_train = X_train[indices]
    y_train = y_train[indices]

    X_train_partitions = []
    y_train_partitions = []

    partition_size = int(len(X_train) / partitions)

    for i in range(partitions):
        if i == partitions - 1:
            X_train_partitions.append(X_train[i * partition_size :])
            y_train_partitions.append(y_train[i * partition_size :])
        else:
            X_train_partitions.append(
                X_train[i * partition_size : (i + 1) * partition_size]
            )
            y_train_partitions.append(
                y_train[i * partition_size : (i + 1) * partition_size]
            )

    return X_train_partitions, y_train_partitions

In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()
X_train, y_train = partition_train_data(X_train, y_train, config["partitions"])

In [7]:
for i in range(len(X_train)):
    np.save(f"../../../data/X_train_{i + 1}.npy", X_train[i])
    np.save(f"../../../data/y_train_{i + 1}.npy", y_train[i])

In [8]:
model = create_model()
rain = Rain(config, model, X_train, y_train)

2023-07-02 20:52:16,275 [DEBUG] [Rain] Rain is initialized
2023-07-02 20:52:16,277 [DEBUG] [Provisioner] Creating coordinator
2023-07-02 20:52:16,280 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-02 20:52:16,306 [DEBUG] [LocalProvisioner] LocalProvisioner is initialized


In [9]:
model = rain.train_centralized_async()

2023-07-02 20:52:16,335 [DEBUG] [Rain] Creating workers
2023-07-02 20:52:16,359 [INFO] [Provisioner] provisioner is serving
2023-07-02 20:52:16,364 [DEBUG] [Provisioner] Starting coordinator
2023-07-02 20:52:16,368 [INFO] [Coordinator] coordinator is serving
2023-07-02 20:52:16,370 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-02 20:52:16,383 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-02 20:52:16,386 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-02 20:52:16,389 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-02 20:52:16,394 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-02 20:52:16,399 [INFO] [Worker_50152] Worker is running on port: 50152
2023-07-02 20:52:16,404 [INFO] [Worker_50153] Worker is running on port: 50153
2023-07-02 20:52:16,406 [DEBUG] [Provisioner] [Created workers]
IPs : ['127.0.0.1', '127.0.0.1', '127.0.0

Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 2s 15ms/step - loss: 0.3020 - accuracy: 0.9095
sending data to coordinator
sending data to coordinator


2023-07-02 20:53:30,899 [DEBUG] [DividerAmbassador] divider received: Executed! for worker2
2023-07-02 20:53:30,913 [DEBUG] [DividerAmbassador] divider begins downloading ../../../Divider/divider/data/2_2_trained.pkl from worker2
2023-07-02 20:53:30,914 [DEBUG] [DividerAmbassador] divider received: Executed! for worker1
2023-07-02 20:53:30,914 [DEBUG] [DividerAmbassador] divider received: Executed! for worker3
2023-07-02 20:53:30,917 [DEBUG] [DividerAmbassador] divider begins downloading ../../../Divider/divider/data/1_1_trained.pkl from worker1
2023-07-02 20:53:30,919 [DEBUG] [DividerAmbassador] divider begins downloading ../../../Divider/divider/data/3_3_trained.pkl from worker3
2023-07-02 20:53:31,486 [DEBUG] [DividerAmbassador] Downloaded ../../../Divider/divider/data/2_2_trained.pkl in divider
2023-07-02 20:53:31,497 [DEBUG] [DividerAmbassador] Downloaded ../../../Divider/divider/data/1_1_trained.pkl in divider
2023-07-02 20:53:31,520 [DEBUG] [DividerAmbassador] Downloaded ../../.

Error in loading the data:  Layer 'sequential' expected 0 variables, but received 1 variables during loading. Expected: []
Error in receiving the data:  'NoneType' object is not subscriptable
Error in loading the data:  Layer 'sequential' expected 0 variables, but received 1 variables during loading. Expected: []
Error in receiving the data:  'NoneType' object is not subscriptable
Error in loading the data:  Layer 'sequential' expected 0 variables, but received 1 variables during loading. Expected: []
Error in receiving the data:  'NoneType' object is not subscriptable


2023-07-02 20:54:27,450 [DEBUG] [DividerAmbassador] divider received: Executed! for worker1
2023-07-02 20:54:27,451 [DEBUG] [DividerAmbassador] divider begins downloading ../../../Divider/divider/data/1_1_trained.pkl from worker1
2023-07-02 20:54:27,499 [DEBUG] [DividerAmbassador] divider received: Executed! for worker3
2023-07-02 20:54:27,503 [DEBUG] [DividerAmbassador] divider begins downloading ../../../Divider/divider/data/3_3_trained.pkl from worker3
2023-07-02 20:54:27,574 [DEBUG] [DividerAmbassador] divider received: Executed! for worker2
2023-07-02 20:54:27,576 [DEBUG] [DividerAmbassador] divider begins downloading ../../../Divider/divider/data/2_2_trained.pkl from worker2
2023-07-02 20:54:27,869 [DEBUG] [DividerAmbassador] Downloaded ../../../Divider/divider/data/1_1_trained.pkl in divider
Exception in thread Thread-19:
Traceback (most recent call last):
  File "/usr/lib/python3.8/threading.py", line 932, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.8/threading.

In [10]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 2ms/step - loss: 0.4282 - accuracy: 0.9352

Test accuracy: 93.5%


In [11]:
# model = rain.train_centralized_sync()

In [12]:
# X_test, y_test = get_test_data()
# loss, acc = model.evaluate(X_test, y_test, batch_size=config["batch_size"])
# print("\nTest accuracy: %.1f%%" % (100.0 * acc))